# Tutorial 2 — Finance Models

spxa is purpose-built for quantitative finance. This notebook covers:
- Exponential Lévy models for asset prices
- European option pricing via the Carr-Madan FFT method
- Implied volatility smile from VG and NIG models
- Comparing models via Hellinger and L² distances

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from spxa.zoo.levy import BrownianMotion, VarianceGamma, NIG, GammaProcess
from spxa.analytics import cumulant_table, l2_char_func_distance, hellinger_distance
from spxa.analytics.fourier import european_call_price, implied_volatility
from spxa.sim import geometric_levy

## 1. Geometric Lévy models

The exponential Lévy model is $S_t = S_0 \exp(rt + X_t)$ where $X$ is a Lévy process.
The risk-neutral drift is chosen so $\mathbb{E}[S_t] = S_0 e^{rt}$.

In [ ]:
rng = np.random.default_rng(0)
S0, r, T = 100.0, 0.05, 1.0

bm_gbm = BrownianMotion(mu=0.0, sigma=0.2)
vg     = VarianceGamma(sigma=0.2, nu=0.1, theta=-0.1)
nig    = NIG(alpha=15.0, beta=-5.0, delta=0.5)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, proc, name in zip(axes, [bm_gbm, vg, nig], ['GBM', 'Variance Gamma', 'NIG']):
    paths = geometric_levy(proc, mu=r, sigma=1.0, s0=S0, T=T, n_steps=100, n_paths=8, rng=rng)
    for i in range(8):
        ax.plot(np.linspace(0, T, 101), paths[i], lw=0.7, alpha=0.8)
    ax.axhline(S0, color='k', lw=0.5, ls='--')
    ax.set_title(name); ax.set_xlabel('Time'); ax.set_ylabel('$S_t$')
plt.tight_layout()
plt.savefig('levy_paths.png', dpi=100)
plt.show()

## 2. European option pricing — Carr-Madan FFT

For a Lévy log-return process $X$ with characteristic function $\phi(u;t)$,
the call price is given by the Carr-Madan formula:
$$
C(K) = \frac{e^{-\alpha k}}{\pi} \int_0^\infty e^{-iuk} \psi(u)\, du
$$
where $k = \log(K/S_0)$ and $\psi$ is a modified version of $\phi$.

In [ ]:
strikes = np.arange(85, 120, 5.0)
models = [
    ('GBM (σ=0.2)',  BrownianMotion(mu=0.0, sigma=0.2)),
    ('VG',           VarianceGamma(sigma=0.2, nu=0.1, theta=-0.1)),
    ('NIG',          NIG(alpha=15.0, beta=-5.0, delta=0.5)),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, proc in models:
    prices = [european_call_price(proc, S0=S0, K=K, r=r, t=T) for K in strikes]
    ivs    = [implied_volatility(p, S0=S0, K=K, r=r, t=T) for p, K in zip(prices, strikes)]
    axes[0].plot(strikes, prices, label=name, marker='o', ms=4)
    axes[1].plot(strikes, ivs,    label=name, marker='o', ms=4)

axes[0].set_title('Call prices'); axes[0].set_xlabel('Strike'); axes[0].legend()
axes[1].set_title('Implied vol smile'); axes[1].set_xlabel('Strike'); axes[1].set_ylabel('Impl. vol')
plt.tight_layout()
plt.savefig('iv_smile.png', dpi=120)
plt.show()

## 3. Cumulant comparison

The cumulants directly encode the shape of the distribution — skewness and kurtosis are why Lévy models produce smiles.

In [ ]:
for name, proc in models:
    t = cumulant_table(proc, order=4)
    print(f'{name:20}  skew={t["skewness"]:+.4f}  kurt={t["excess_kurtosis"]:+.4f}')

## 4. Model distances

spxa computes L² and Hellinger distances between marginal distributions directly from the characteristic functions — no Monte Carlo needed.

In [ ]:
bm_ref = BrownianMotion(mu=0.0, sigma=0.2)
vg_sym = VarianceGamma(sigma=0.2, nu=0.1, theta=0.0)
print('L²(GBM, VG)  :', l2_char_func_distance(bm_ref, vg_sym, u_max=20, n_points=500))
print('H²(GBM, VG)  :', hellinger_distance(bm_ref, vg_sym, u_max=20, n_points=500))
print('L²(GBM, GBM) :', l2_char_func_distance(bm_ref, bm_ref, u_max=20, n_points=500), ' (expected ≈0)')

## 5. Stochastic volatility via subordination

The Variance Gamma model is exactly a BM subordinated by a Gamma process. The subordinator acts as a stochastic clock — business time vs calendar time.

In [ ]:
bm_sub = BrownianMotion(mu=-0.1, sigma=0.2)
g_sub  = GammaProcess(a=1.0/0.1, b=1.0/0.1)  # IG(1/nu, 1/nu) for VG
W_sub  = bm_sub @ g_sub

rng2 = np.random.default_rng(1)
paths_sub = W_sub.simulate(n_steps=252, n_paths=5000, T=1.0, rng=rng2)
vg_ref     = VarianceGamma(sigma=0.2, nu=0.1, theta=-0.1)
paths_vg   = vg_ref.simulate(n_steps=252, n_paths=5000, T=1.0, rng=rng2)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, p, label in zip(axes, [paths_sub[:,-1], paths_vg[:,-1]], ['BM@Gamma', 'VG direct']):
    ax.hist(p, bins=40, density=True, alpha=0.7)
    ax.set_title(label); ax.set_xlabel('$X_1$')
print('BM@Gamma mean:', paths_sub[:,-1].mean().round(4), '  VG mean:', paths_vg[:,-1].mean().round(4))
plt.tight_layout()
plt.savefig('subordination_hist.png', dpi=120)
plt.show()